In [2]:
%pip install pvlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# AI for building energy: forecasting solar and demand to run a battery smarter
**Author:** Maduakor Uchechukwu David · MSc Energy for Solar Buildings and Cities

**Business question:** a commercial building in Grenoble has a 400 kWp rooftop PV plant and a 300 kWh battery. Can machine-learning (ML) forecasts of tomorrow's solar output and building demand reduce its electricity bill, compared with simple rules?

| # | Step | Method |
|---|---|---|
| 1 | Data | 2 years of hourly PVGIS weather, PV output with `pvlib`, office demand model |
| 2 | Forecasting | Day-ahead forecasts: persistence baseline vs gradient-boosted trees (`scikit-learn`) |
| 3 | Explainability | Permutation importance: which inputs drive each forecast |
| 4 | Battery control | Same battery, five control strategies |
| 5 | Business case | Annual cost, self-consumption, self-sufficiency, CO₂ |

Year 1 (2019) trains the models. Year 2 (2020) is kept unseen for testing, which is how a forecast is judged in a real deployment.

In [3]:
##libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error

RNG = np.random.default_rng(7)     

## 1. Data
### 1.1 Site, PV system and weather
Grenoble (45.19 N, 5.72 E). Rooftop PV: **400 kWp**, 15° tilt facing south, 330 kW of inverters.
Hourly **in-plane irradiance**, air temperature and wind speed for **2019–2020** come from the PVGIS API (SARAH satellite data).
Unlike the TMY used in notebook 1, these are **real consecutive years**, which a forecast needs.

In [4]:
#A: site settings

SITE = dict(name="Grenoble, France", lat=45.19, lon=5.72, tz="Europe/Paris")
PV_KWP, INV_KW, TILT, AZIMUTH = 400, 330, 15, 180
YEARS = (2019, 2020)
location = pvlib.location.Location(SITE["lat"], SITE["lon"], tz="UTC", altitude=212)

In [8]:
#B: download two real years from PVGIS

raw, *_ = pvlib.iotools.get_pvgis_hourly(
    SITE["lat"], SITE["lon"], start=YEARS[0], end=YEARS[1],
    surface_tilt=TILT, surface_azimuth=AZIMUTH, components=True,
    map_variables=True, timeout=60)

weather = pd.DataFrame({
    "poa_global": raw[["poa_direct", "poa_sky_diffuse", "poa_ground_diffuse"]].sum(axis=1),
    "temp_air":   raw["temp_air"],
    "wind_speed": raw["wind_speed"]})
weather.index = weather.index.floor("h") + pd.Timedelta("30min")

print(f"{len(weather):,} hours from {weather.index[0]} to {weather.index[-1]}")
weather.describe().round(1)

17,544 hours from 2019-01-01 00:30:00+00:00 to 2020-12-31 23:30:00+00:00


,poa_global,temp_air,wind_speed
count,17544.0,17544.0,17544.0
mean,186.2,9.0,1.1
std,286.8,8.3,0.5
min,0.0,-11.5,0.0
25%,0.0,2.5,0.7
50%,0.0,8.6,1.0
75%,293.3,15.1,1.4
max,1059.2,31.6,3.7


In [9]:
#C: PV output

params = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS["sapm"]["close_mount_glass_glass"]
t_cell = pvlib.temperature.sapm_cell(weather["poa_global"], weather["temp_air"],
                                     weather["wind_speed"], **params)

pdc = pvlib.pvsystem.pvwatts_dc(weather["poa_global"], t_cell, PV_KWP, -0.0035) * 0.86   # 14 % losses
pv  = (pdc * 0.975).clip(lower=0, upper=INV_KW).rename("pv_kw")                         # inverter 97.5 %, 330 kW cap

annual = pv.sum() / len(YEARS)
print(f"Annual PV ≈ {annual/1000:,.0f} MWh  |  {annual/PV_KWP:,.0f} kWh/kWp  |  peak {pv.max():.0f} kW")

Annual PV ≈ 509 MWh  |  1,272 kWh/kWp  |  peak 308 kW


**Data check:** 17,544 hours (2019–2020, including the 2020 leap day), with no missing values.
- Mean in-plane irradiance 186 W/m² → about **1,630 kWh/m²/year** on the 15° plane.
- PV output: **≈509 MWh/year (1,272 kWh/kWp)**. The peak of 308 kW stays below the 330 kW inverter limit, so there is no clipping.
- Mean air temperature 9.0 °C and wind speed 1.1 m/s are **lower than Grenoble city measurements** (≈12–13 °C). PVGIS uses reanalysis data (ERA5) averaged over a grid cell that includes the surrounding mountains. This is a known limitation of the data, and it slightly underestimates cell temperature.